# TCC WikiArt — Etapa 11: Interpretabilidade com Grad-CAM (ResNet-50)

Gera mapas Grad-CAM por patch e por pintura para exemplos **selecionados de forma reproduzível** do conjunto de teste (acertos e erros por classe, com foco em Renaissance × Baroque, `religious_painting` e `portrait`).

Nenhum treinamento é feito. Os mapas indicam regiões que aumentaram a pontuação da classe-alvo; **não provam compreensão semântica** nem causalidade.

## 1. Ambiente e dependências

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
%cd /content

!rm -rf tcc_wikiart
!git clone https://github.com/fbrum44/tcc_wikiart.git

%cd /content/tcc_wikiart

In [ ]:
!git log -1 --oneline
!git status --short
!ls src/interpretability

In [ ]:
!pip install -q datasets opencv-python scikit-learn timm pytest

In [ ]:
%env HF_HOME=/content/hf_cache

## 2. Dataset WikiArt

In [ ]:
from datasets import load_dataset

wikiart = load_dataset("huggan/wikiart", split="train", keep_in_memory=False)
print("Imagens WikiArt:", len(wikiart))

if len(wikiart) != 81444:
    raise RuntimeError(f"Esperadas 81.444 imagens, encontradas {len(wikiart)}.")

## 3. Google Drive, checkpoint e predições da Etapa 9

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os, shutil
from pathlib import Path

REPO_DIR = Path("/content/tcc_wikiart")
DRIVE_ROOT = Path("/content/drive/MyDrive/TCC_WikiArt_Compartilhado")
MODELS_DRIVE = DRIVE_ROOT / "models"
LOCAL_MODELS = REPO_DIR / "models"

print("Pasta compartilhada:", DRIVE_ROOT.exists())
print("Checkpoint ResNet:", (MODELS_DRIVE / "resnet50_finetune_best.pt").exists())

if LOCAL_MODELS.is_symlink():
    LOCAL_MODELS.unlink()
elif LOCAL_MODELS.exists():
    shutil.rmtree(LOCAL_MODELS)
os.symlink(MODELS_DRIVE, LOCAL_MODELS, target_is_directory=True)
print("models ->", LOCAL_MODELS.resolve())

In [ ]:
# As predições da Etapa 9 podem não estar no GitHub; se faltarem, copia do Drive.
NECESSARIOS = [
    "test_painting_predictions.csv.gz",   # obrigatório (seleção dos exemplos)
    "test_patch_predictions.csv.gz",      # opcional (checagem de consistência por patch)
]
destino = REPO_DIR / "outputs" / "evaluation" / "resnet50"
destino.mkdir(parents=True, exist_ok=True)

for nome in NECESSARIOS:
    alvo = destino / nome
    if alvo.exists():
        print("No repositório:", nome)
        continue
    origem = DRIVE_ROOT / "outputs" / "evaluation" / "resnet50" / nome
    if origem.exists():
        shutil.copy2(origem, alvo)
        print("Copiado do Drive:", nome)
    else:
        print("NÃO ENCONTRADO:", nome)

assert (destino / "test_painting_predictions.csv.gz").exists(), "Faltam as predições por pintura da Etapa 9."

## 4. Testes automatizados

Testam reconstrução do mapa, métricas, descritores e seleção reproduzível; e o Grad-CAM com uma ResNet-50 sem pesos pré-treinados (verifica formato 8×8, valores não negativos e que os logits coincidem com a inferência normal).

In [ ]:
!python -m pytest tests/test_gradcam_mapas.py -q
!python -m src.interpretability.gradcam

## 5. Smoke test (2 pinturas)

Confere o fluxo completo antes da execução oficial. Os resultados vão para uma pasta separada.

In [ ]:
!python -u -m src.interpretability.run_gradcam \
    --checkpoint models/resnet50_finetune_best.pt \
    --out outputs/gradcam/resnet50_smoke \
    --max-pinturas 2

In [ ]:
import json
from pathlib import Path

print(json.dumps(json.loads(Path("outputs/gradcam/resnet50_smoke/verificacao.json").read_text()), indent=2, ensure_ascii=False))

from IPython.display import Image, display
for f in sorted(Path("outputs/gradcam/resnet50_smoke/paintings").glob("*.png"))[:2]:
    display(Image(filename=str(f)))

**Antes de seguir:** em `verificacao.json`, `concordancia_patch_media` deve ficar muito próxima de 1,0 e `predicao_pintura_reproduzida` deve ser 1,0. Isso confirma que o Grad-CAM usa o mesmo modelo e as mesmas entradas da avaliação.

## 6. Execução oficial

Cerca de 40 a 50 pinturas × ~140 patches. Para cada pintura selecionada gera-se o mapa da classe prevista (e da classe real quando o modelo errou).

In [ ]:
!python -u -m src.interpretability.run_gradcam \
    --checkpoint models/resnet50_finetune_best.pt \
    --out outputs/gradcam/resnet50

## 7. Verificação e resultados

In [ ]:
import json
from pathlib import Path
import pandas as pd

OUT = Path("outputs/gradcam/resnet50")
print(json.dumps(json.loads((OUT / "verificacao.json").read_text()), indent=2, ensure_ascii=False))

sel = pd.read_csv(OUT / "selecao.csv")
print()
print(sel.groupby(["true_class", "acertou"]).size().unstack(fill_value=0))

In [ ]:
pd.read_csv(OUT / "resumo_por_grupo.csv")

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(OUT / "resumo_distribuicao.png")))
for f in sorted((OUT / "montagens").glob("*.png")):
    print(f.name)
    display(Image(filename=str(f)))

## 8. Backup no Drive e pacote para o GitHub

Os resultados ficam no Drive compartilhado. O `.zip` contém tudo o que será versionado (figuras, CSVs e os mapas 8×8 em `.npz`); baixe-o e extraia em `outputs/gradcam/` no repositório.

In [ ]:
import shutil
from pathlib import Path

DESTINO = DRIVE_ROOT / "outputs" / "gradcam" / "resnet50"
if DESTINO.exists():
    shutil.rmtree(DESTINO)
shutil.copytree(OUT, DESTINO)
print("Backup completo no Drive:", DESTINO)

# Pacote leve para o GitHub: sem as figuras individuais (paintings/ e patches/), que são grandes.
leve = Path("/content/gradcam_leve/resnet50")
if leve.parent.exists():
    shutil.rmtree(leve.parent)
shutil.copytree(OUT, leve, ignore=shutil.ignore_patterns("paintings", "patches"))
shutil.make_archive("/content/gradcam_resnet50_leve", "zip", root_dir="/content/gradcam_leve", base_dir="resnet50")
shutil.copy2("/content/gradcam_resnet50_leve.zip", DRIVE_ROOT / "outputs" / "gradcam_resnet50_leve.zip")
print("ZIP leve:", DRIVE_ROOT / "outputs" / "gradcam_resnet50_leve.zip")
!du -sh /content/gradcam_resnet50_leve.zip outputs/gradcam/resnet50

## 9. Execução ampliada (para comparar acertos e erros)

Com poucas pinturas por grupo (4 a 7) não é possível afirmar diferenças entre acertos e erros. Esta execução usa até 20 pinturas por grupo, **sem salvar as figuras individuais** (arquivos grandes), e a análise estatística lê só `metricas_por_pintura.csv`. A seleção continua determinística (SHA-256, seed 42); os grupos pequenos (por exemplo, erros de Cubism) usam todas as pinturas disponíveis.

In [ ]:
!python -u -m src.interpretability.run_gradcam \
    --checkpoint models/resnet50_finetune_best.pt \
    --out outputs/gradcam/resnet50_amplo \
    --n-por-grupo 20 \
    --sem-figuras-individuais

In [ ]:
!python -m src.interpretability.analise_grupos --in outputs/gradcam/resnet50_amplo

In [ ]:
import shutil
from pathlib import Path

OUT2 = Path("outputs/gradcam/resnet50_amplo")
leve = Path("/content/gradcam_leve2/resnet50_amplo")
if leve.parent.exists():
    shutil.rmtree(leve.parent)
shutil.copytree(OUT2, leve, ignore=shutil.ignore_patterns("paintings", "patches"))
shutil.make_archive("/content/gradcam_resnet50_amplo_leve", "zip", root_dir="/content/gradcam_leve2", base_dir="resnet50_amplo")
shutil.copy2("/content/gradcam_resnet50_amplo_leve.zip", DRIVE_ROOT / "outputs" / "gradcam_resnet50_amplo_leve.zip")
DEST2 = DRIVE_ROOT / "outputs" / "gradcam" / "resnet50_amplo"
if DEST2.exists():
    shutil.rmtree(DEST2)
shutil.copytree(OUT2, DEST2)
print("Backup:", DEST2)
!du -sh /content/gradcam_resnet50_amplo_leve.zip